In [ ]:
import scvi
import scanpy as sc
import scipy.sparse as sp
import pandas as pd
from scvi.external import MRVI
import os
import tempfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scvi
import seaborn as sns
from scvi.external import MRVI


# load your AnnData object



In [ ]:
import scvi
import scanpy as sc
import scipy.sparse as sp
import pandas as pd
from scvi.external import MRVI
import os
import tempfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scvi
import seaborn as sns
from scvi.external import MRVI


# load your AnnData object



In [ ]:
import torch

print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA version: {torch.version.cuda}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

In [ ]:

X = sc.read_mtx("/home/nlapique/Blood/count_matrix.mtx").X       # sparse matrix
X = X.T
# Load gene and cell names


In [ ]:

genes = pd.read_csv("/home/nlapique/Blood/genes.tsv", header=None)[0].values
cells = pd.read_csv("/home/nlapique/Blood/barcodes.tsv", header=None)[0].values

# Transpose to match AnnData's expected orientation


adata = sc.AnnData(X=X)
adata.var_names = genes     # now length = n_vars
adata.obs_names = cells     # now length = n_obs



In [ ]:
sc.pp.highly_variable_genes(
    adata,
    n_top_genes=3000,
    flavor="seurat_v3"
)
adata = adata[:, adata.var["highly_variable"]].copy()

In [ ]:
meta = pd.read_csv(
    "/home/nlapique/Blood/features.txt",
    sep="\t",
    index_col=0    # <-- IMPORTANT: use first column (cell barcode) as index
)
meta.head()
meta.index


In [ ]:
adata.obs = adata.obs.join(meta, how="left")
    
adata.obs.head()



In [ ]:
import matplotlib.pyplot as plt
import numpy as np

patient_col = "replicate"
sex_col = "sex"
male_code = 2   # <-- MEN

variables = [
    "bmi26",
    "MeanArtPr26",
    "VOmax26",
    "choles26",
    "trigly26",
    "hdl26",
    "urea26",
    "crp26ns",
]

quantiles = [0.20, 0.60, 0.80, 0.99]

# --------------------------------------------
# Restrict to MEN only
# --------------------------------------------

adata_male = adata[adata.obs[sex_col] == male_code].copy()

print("Number of male cells:", adata_male.n_obs)
print("Number of male patients:", adata_male.obs[patient_col].nunique())

# --------------------------------------------
# Loop over variables
# --------------------------------------------

for var in variables:

    patient_values = (
        adata_male.obs[[patient_col, var]]
        .dropna()
        .drop_duplicates(subset=[patient_col])
        .set_index(patient_col)[var]
        .astype(float)
    )

    print(f"\n{var}")
    print("Number of male patients:", patient_values.shape[0])

    if patient_values.shape[0] < 10:
        print("⚠️ Too few male patients — skipping")
        continue

    q20, q60, q80, q99 = patient_values.quantile(quantiles)

    print(
        f"q20={q20:.2f}, q60={q60:.2f}, "
        f"q80={q80:.2f}, q99={q99:.2f}"
    )

    # Histogram
    plt.figure(figsize=(8, 5))
    plt.hist(patient_values, bins=30)

    plt.axvline(q20, linestyle="--", linewidth=2)
    plt.axvline(q60, linestyle="--", linewidth=2)
    plt.axvline(q80, linestyle="--", linewidth=2)
    plt.axvline(q99, linestyle="--", linewidth=2)

    plt.xlabel(var)
    plt.ylabel("Number of male patients")
    plt.title(f"{var} distribution (men only)")

    ymax = plt.ylim()[1]
    plt.text(q20, ymax * 0.9, "q20", rotation=90, va="top")
    plt.text(q60, ymax * 0.9, "q60", rotation=90, va="top")
    plt.text(q80, ymax * 0.9, "q80", rotation=90, va="top")
    plt.text(q99, ymax * 0.9, "q99", rotation=90, va="top")

    plt.tight_layout()
    plt.show()


In [ ]:
import numpy as np

for var in variables:

    patient_values = (
        adata_male.obs[[patient_col, var]]
        .dropna()
        .drop_duplicates(subset=[patient_col])
        .set_index(patient_col)[var]
        .astype(float)
    )

    if patient_values.shape[0] < 10:
        print(f"{var}: too few male patients, skipped")
        continue

    q20, q60, q80, q99 = patient_values.quantile(quantiles)

    def bin_func(x):
        if q20 <= x <= q60:
            return f"{var}_q20_60_male"
        elif q80 <= x <= q99:
            return f"{var}_q80_99_male"
        else:
            return np.nan

    # Patient-level bins (men only)
    patient_bins = patient_values.apply(bin_func)

    # Map back to ALL cells
    adata_male.obs[f"{var}_bin_male"] = (
    adata_male.obs[patient_col]
    .map(patient_bins)
    .astype("category")
)

    print(
        f"{var}_bin_male added | "
        f"control={(patient_bins == f'{var}_q20_60_male').sum()} | "
        f"test={(patient_bins == f'{var}_q80_99_male').sum()}"
    )


In [ ]:
import numpy as np

# Define quantiles once
quantiles = [0.30, 0.70, 0.70, 1]

for var in variables:

    patient_values = (
        adata_male.obs[[patient_col, var]]
        .dropna()
        .drop_duplicates(subset=[patient_col])
        .set_index(patient_col)[var]
        .astype(float)
    )

    if patient_values.shape[0] < 10:
        print(f"{var}: too few male patients, skipped")
        continue

    q20, q60, q80, q99 = patient_values.quantile(quantiles)

    # -----------------------------
    # Define binning function
    # -----------------------------
    def bin_func(x):
        if x <= q20:
            return f"{var}_q0_20_male"
        elif q20 < x <= q60:
            return f"{var}_q20_60_male"
        elif q80 < x <= q99:
            return f"{var}_q80_99_male"
        else:
            return np.nan

    # -----------------------------
    # Patient-level bins
    # -----------------------------
    patient_bins = patient_values.apply(bin_func)

    # -----------------------------
    # Map back to ALL cells
    # -----------------------------
    adata_male.obs[f"{var}_bin_male"] = (
        adata_male.obs[patient_col]
        .map(patient_bins)
        .astype("category")
    )

    # -----------------------------
    # Reporting
    # -----------------------------
    print(
        f"{var}_bin_male added | "
        f"q0_20={(patient_bins == f'{var}_q0_20_male').sum()} | "
        f"q20_60={(patient_bins == f'{var}_q20_60_male').sum()} | "
        f"q80_99={(patient_bins == f'{var}_q80_99_male').sum()}"
    )


In [ ]:
test_adata=adata_male


In [ ]:
import matplotlib.pyplot as plt

# Count cells per patient
cell_counts = test_adata.obs["replicate"].value_counts().sort_values(ascending=False)

print(cell_counts.describe())

plt.figure(figsize=(10, 4))
plt.bar(range(len(cell_counts)), cell_counts.values)
plt.xlabel("Patients (replicates)")
plt.ylabel("Number of cells")
plt.title("Cell count per patient (sorted)")
plt.tight_layout()
plt.show()


In [ ]:
# Count cells per patient
cell_counts = test_adata.obs["replicate"].value_counts()

# Keep only patients with >= 10 cells
min_cells = 10
good_replicates = cell_counts[cell_counts >= min_cells].index

# Subset AnnData
test_adata = test_adata[test_adata.obs["replicate"].isin(good_replicates)].copy()

# Sanity checks
print("Remaining patients:", test_adata.obs["replicate"].nunique())
print("Remaining cells:", test_adata.n_obs)


In [ ]:
expected_categories = [
    "VOmax26_q0_20_male",
    "VOmax26_q20_60_male",
    "VOmax26_q80_99_male",
]

test_adata.obs["VOmax26_bin_male"] = (
    test_adata.obs["VOmax26_bin_male"]
    .astype("category")
    .cat.set_categories(expected_categories)
)


In [ ]:
import pandas as pd
import numpy as np

# Check current state
print("Current VOmax26_bin_male dtype:", test_adata.obs['VOmax26_bin_male'].dtype)
print("Unique values:", test_adata.obs['VOmax26_bin_male'].unique())
print("Value counts:")
print(test_adata.obs['VOmax26_bin_male'].value_counts(dropna=False))

# Fix 1: Remove any NaN values first
print("\nRemoving NaN values...")
test_adata = test_adata[~test_adata.obs['VOmax26_bin_male'].isna()].copy()
print(f"After removing NaN: {test_adata.n_obs} cells")

# Fix 2: Convert to categorical with explicit categories
print("\nConverting to categorical...")
test_adata.obs['VOmax26_bin_male'] = pd.Categorical(
    test_adata.obs['VOmax26_bin_male'],
    categories=['VOmax26_q0_20_male', 'VOmax26_q20_60_male', 'VOmax26_q80_99_male']
)

print("After conversion:")
print("  Dtype:", test_adata.obs['VOmax26_bin_male'].dtype)
print("  Categories:", test_adata.obs['VOmax26_bin_male'].cat.categories)
print("  Value counts:")
print(test_adata.obs['VOmax26_bin_male'].value_counts())

# Now setup MRVI
from scvi.external import MRVI

print("\nSetting up MRVI...")
MRVI.setup_anndata(
    test_adata,
    sample_key="VOmax26_bin_male",
    backend="torch"
)

print("✓ Setup successful!")

In [ ]:
from scipy.sparse import csr_matrix

if not isinstance(test_adata.X, csr_matrix):
    test_adata.X = csr_matrix(test_adata.X)


In [ ]:
import os
import torch

# Check GPU status
print("Checking GPU availability...")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Number of GPUs: {torch.cuda.device_count()}")

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}: {torch.cuda.get_device_name(i)}")
        print(f"  Memory allocated: {torch.cuda.memory_allocated(i) / 1e9:.2f} GB")
        print(f"  Memory reserved: {torch.cuda.memory_reserved(i) / 1e9:.2f} GB")

# Solution 1: Clear GPU memory
print("\nClearing GPU cache...")
torch.cuda.empty_cache()

In [ ]:
import scvi
import lightning
import torch

print("scvi-tools:", scvi.__version__)
print("lightning:", lightning.__version__)
print("torch:", torch.__version__)


In [ ]:
test_adata.write("adata_men_removed_donors2.h5ad")


In [ ]:

import scvi
import scanpy as sc
import pandas as pd
import numpy as np
from scvi.external.mrvi_torch import TorchMRVI
import os
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

# Force PyTorch backend
os.environ["SCVI_USE_TORCH"] = "1"

# Load your data
print("Loading data...")
test_adata = sc.read_h5ad('/home/nlapique/Blood/adata_men_removed_donors2.h5ad')
test_adata.X = test_adata.X.tocsr()

sample_key = "VOmax26_bin_male"

# Configuration
N_MODELS = 6  # Number of models to train
SEEDS = [42, 123, 456, 789, 2024, 3141] 
BASE_DIR = "/home/nlapique/Blood/mrvi_ensemble"
os.makedirs(BASE_DIR, exist_ok=True)

In [ ]:


# Training parameters (improved from your working config)
TRAIN_PARAMS = {
    "max_epochs": 1000,  # Increased from 600
    "accelerator": "gpu",
    "devices": 1,
    "precision": "16-mixed",
    "batch_size": 1024,
    "train_size": 0.9,
    "datasplitter_kwargs": {
        "num_workers": 14,
        "pin_memory": True,
        "persistent_workers": True,
    },
    "plan_kwargs": {
        "n_epochs_kl_warmup": 100,  # Increased from 50 for better convergence
    },
    "check_val_every_n_epoch": 5,
}

print("\n" + "="*70)
print("ENSEMBLE MRVI TRAINING")
print("="*70)
print(f"Training {N_MODELS} models with seeds: {SEEDS}")
print(f"Max epochs: {TRAIN_PARAMS['max_epochs']}")
print(f"KL warmup: {TRAIN_PARAMS['plan_kwargs']['n_epochs_kl_warmup']}")
print(f"Batch size: {TRAIN_PARAMS['batch_size']}")
print(f"Workers: {TRAIN_PARAMS['datasplitter_kwargs']['num_workers']}")

# Store models and their outputs
models = []
denoised_outputs = []
training_histories = []

# Train each model
for i, seed in enumerate(SEEDS):
    print("\n" + "="*70)
    print(f"TRAINING MODEL {i+1}/{N_MODELS} (seed={seed})")
    print("="*70)
    
    # Setup - matching your working config exactly
    print(f"\nSetting up model with seed {seed}...")
    TorchMRVI.setup_anndata(test_adata, sample_key=sample_key)
    
    # Create model with seed for reproducibility
    model = TorchMRVI(test_adata)
    
    # Note: TorchMRVI doesn't expose a seed parameter directly,
    # but PyTorch's random state will differ between runs
    # Set PyTorch seed manually
    import torch
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    
    # Train - using your exact working parameters
    print(f"Training model {i+1}...")
    model.train(**TRAIN_PARAMS)
    
    # Get training history if available
    try:
        history_df = model.history
        # Convert DataFrame to dict with lists
        history = {}
        if 'elbo_train' in history_df.columns:
            history['elbo_train'] = history_df['elbo_train'].values.tolist()
        else:
            history['elbo_train'] = []
        
        if 'elbo_validation' in history_df.columns:
            history['elbo_validation'] = history_df['elbo_validation'].values.tolist()
        else:
            history['elbo_validation'] = []
    except Exception as e:
        print(f"  Note: Could not extract training history: {e}")
        history = {'elbo_train': [], 'elbo_validation': []}
    
    # Save individual model
    model_path = f"{BASE_DIR}/model_{i+1}_seed_{seed}"
    model.save(model_path, overwrite=True)
    print(f"✓ Saved model to: {model_path}")
    
    # Get denoised expression
    print(f"Extracting denoised expression from model {i+1}...")
    denoised = model.get_normalized_expression()
    
    # Store results
    models.append(model)
    denoised_outputs.append(denoised)
    training_histories.append(history)
    
    print(f"✓ Model {i+1} complete")
    if len(history.get('elbo_train', [])) > 0:
        print(f"  Final ELBO: {history['elbo_train'][-1]:.2f}")
        print(f"  Epochs trained: {len(history['elbo_train'])}")
    else:
        print(f"  Training completed (history not available)")

print("\n" + "="*70)
print("ENSEMBLE AVERAGING")
print("="*70)

# Convert to numpy arrays for easier manipulation
denoised_arrays = [d.toarray() if hasattr(d, 'toarray') else np.array(d) 
                   for d in denoised_outputs]

# Calculate ensemble average
print("Calculating ensemble average...")
ensemble_denoised = np.mean(denoised_arrays, axis=0)

# Calculate ensemble variance (uncertainty)
ensemble_variance = np.var(denoised_arrays, axis=0)
ensemble_std = np.sqrt(ensemble_variance)

print(f"✓ Ensemble denoised shape: {ensemble_denoised.shape}")
print(f"  Mean expression: {ensemble_denoised.mean():.4f}")
print(f"  Std deviation: {ensemble_denoised.std():.4f}")
print(f"  Mean ensemble uncertainty: {ensemble_std.mean():.4f}")

# Add ensemble results to AnnData
print("\nAdding results to AnnData...")
test_adata.layers['mrvi_ensemble_denoised'] = ensemble_denoised
test_adata.layers['mrvi_ensemble_std'] = ensemble_std

# Also save individual model outputs
for i in range(N_MODELS):
    test_adata.layers[f'mrvi_model_{i+1}_denoised'] = denoised_arrays[i]

print("✓ Added layers to AnnData:")
print("  - mrvi_ensemble_denoised (main output)")
print("  - mrvi_ensemble_std (uncertainty)")
for i in range(N_MODELS):
    print(f"  - mrvi_model_{i+1}_denoised")

# Save the updated AnnData
output_file = f"{BASE_DIR}/adata_with_ensemble_denoised.h5ad"
test_adata.write_h5ad(output_file)
print(f"\n✓ Saved AnnData to: {output_file}")

print("\n" + "="*70)
print("QUALITY CONTROL ANALYSIS")
print("="*70)

# Compare models with each other
print("\nModel agreement analysis:")
correlations = []
for i in range(N_MODELS):
    for j in range(i+1, N_MODELS):
        # Calculate correlation between flattened arrays
        flat_i = denoised_arrays[i].flatten()
        flat_j = denoised_arrays[j].flatten()
        corr, _ = pearsonr(flat_i, flat_j)
        correlations.append(corr)
        print(f"  Model {i+1} vs Model {j+1}: r = {corr:.4f}")

mean_agreement = np.mean(correlations)
print(f"\nMean pairwise correlation: {mean_agreement:.4f}")

if mean_agreement > 0.95:
    print("✓ Excellent model agreement!")
elif mean_agreement > 0.90:
    print("✓ Good model agreement")
else:
    print("⚠ Warning: Low model agreement - consider retraining")

# Genes with high uncertainty
print("\nGenes with highest ensemble uncertainty:")
mean_uncertainty_per_gene = ensemble_std.mean(axis=0)
high_uncertainty_genes = np.argsort(mean_uncertainty_per_gene)[-10:]
for idx in high_uncertainty_genes[::-1]:
    gene_name = test_adata.var_names[idx]
    uncertainty = mean_uncertainty_per_gene[idx]
    print(f"  {gene_name}: {uncertainty:.4f}")

print("\n" + "="*70)
print("VISUALIZATION")
print("="*70)

# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

has_history = False
for i, history in enumerate(training_histories):
    elbo_train = history.get('elbo_train', [])
    elbo_val = history.get('elbo_validation', [])
    
    if len(elbo_train) > 0:
        has_history = True
        epochs = range(len(elbo_train))
        axes[0].plot(epochs, elbo_train, label=f'Model {i+1}', alpha=0.7)
        if len(elbo_val) > 0:
            axes[1].plot(range(len(elbo_val)), elbo_val, label=f'Model {i+1}', alpha=0.7)

if has_history:
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Training ELBO')
    axes[0].set_title('Training Curves')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Validation ELBO')
    axes[1].set_title('Validation Curves')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(f"{BASE_DIR}/training_curves.png", dpi=150, bbox_inches='tight')
    print(f"✓ Saved training curves to: {BASE_DIR}/training_curves.png")
    plt.close()
else:
    print("⚠ Training history not available for plotting")
    plt.close()

# Plot model agreement
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter plot comparing two models
axes[0].scatter(denoised_arrays[0].flatten()[::100], 
                denoised_arrays[1].flatten()[::100],
                alpha=0.1, s=1)
axes[0].set_xlabel('Model 1 Expression')
axes[0].set_ylabel('Model 2 Expression')
axes[0].set_title(f'Model Agreement (r={correlations[0]:.3f})')
axes[0].plot([0, denoised_arrays[0].max()], [0, denoised_arrays[1].max()], 
             'r--', alpha=0.5)

# Histogram of ensemble uncertainty
axes[1].hist(ensemble_std.flatten(), bins=100, alpha=0.7, edgecolor='black')
axes[1].set_xlabel('Ensemble Standard Deviation')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Distribution of Ensemble Uncertainty')
axes[1].set_yscale('log')

plt.tight_layout()
plt.savefig(f"{BASE_DIR}/ensemble_quality.png", dpi=150, bbox_inches='tight')
print(f"✓ Saved quality plots to: {BASE_DIR}/ensemble_quality.png")
plt.close()

print("\n" + "="*70)
print("ENSEMBLE TRAINING COMPLETE!")
print("="*70)
print(f"\nOutputs saved to: {BASE_DIR}/")
print("\nKey files:")
print(f"  - adata_with_ensemble_denoised.h5ad (main output)")
print(f"  - model_1_seed_{SEEDS[0]}/ (individual models)")
print(f"  - training_curves.png")
print(f"  - ensemble_quality.png")
print("\nRecommended usage:")
print("  Use 'mrvi_ensemble_denoised' layer for downstream analysis")
print("  Check 'mrvi_ensemble_std' to identify uncertain genes/cells")

In [ ]:
import scvi
import scanpy as sc
import pandas as pd
from scvi.external import MRVI

# Force PyTorch backend
import os
os.environ["SCVI_USE_TORCH"] = "1"

# Or use TorchMRVI directly
from scvi.external.mrvi_torch import TorchMRVI


test_adata.X = test_adata.X.tocsr()
sample_key = "VOmax26_bin_male"

# Setup with PyTorch
TorchMRVI.setup_anndata(test_adata, sample_key=sample_key)
model = TorchMRVI(test_adata)

# Train on GPU with PyTorch
model.train(
    max_epochs=600,
    accelerator="gpu",
    devices=1,
    precision="16-mixed",
    batch_size=1024,
    train_size=0.9,
    datasplitter_kwargs={
        "num_workers": 14,
        "pin_memory": True,
        "persistent_workers": True,
    },
    plan_kwargs={
        "n_epochs_kl_warmup": 50,
    },
    check_val_every_n_epoch=5,
)
# Save to specific directory
model.save("/home/nlapique/Blood/mrvi_blood_model_3000_hvg_donnors_men_500epochs_vo2max2", overwrite=True)
model2=model


In [ ]:
plt.plot(model.history["elbo_validation"].iloc[5:])
plt.xlabel("Epoch")
plt.ylabel("Validation ELBO")
plt.show()

In [ ]:
plt.plot(model.history["elbo_train"])
plt.xlabel("Epoch")
plt.ylabel("Training ELBO")
plt.show()

In [ ]:
u = model.get_latent_representation()
adata.obsm["u"] = u
sc.pp.neighbors(adata, use_rep="u")
sc.tl.umap(adata, min_dist=0.3)

In [ ]:
import sys
print(sys.executable)


In [ ]:
test_adata = test_adata[test_adata.obs["replicate"] != "DN0608"].copy()


In [ ]:
import scanpy as sc
from scvi.external.mrvi_torch import TorchMRVI  # Use the PyTorch version

# Load your AnnData (the model needs the data structure)


# Load the trained model
model = TorchMRVI.load(
    "/home/nlapique/Blood/mrvi_blood_model_all_genes_donnors_men_500epochs",
    adata=test_adata
)

print("✓ Model loaded successfully!")

In [ ]:





# Move to GPU
model.to_device("cuda:0")
# or simply:
model.to_device("cuda")

print("✓ Model moved to GPU")



In [ ]:
# Convert to category
model.sample_info["VOmax26_bin_male"] = (
    model.sample_info["VOmax26_bin_male"].astype("category")
)
# Run DE

import time
import torch
import os

torch.set_num_threads(15)   # safe anytime

model.to_device("cpu")
start = time.time()

de_res = model.differential_expression(
    sample_cov_keys=["VOmax26_bin_male"],
    store_lfc=True,
    n_samples=100, 
    use_vmap=False
)


end = time.time()

print("DE runtime (seconds):", end - start)

In [ ]:
import time
import torch
import os
import scanpy as sc
from scvi.external.mrvi_torch import TorchMRVI

# Set environment
os.environ["CUDA_VISIBLE_DEVICES"] = ""  # Force CPU
torch.set_num_threads(15)

# Identify HVGs
print("Selecting highly variable genes...")
if "highly_variable" not in test_adata.var.columns:
    sc.pp.highly_variable_genes(
        test_adata, 
        n_top_genes=2000,  # Adjust this: 500, 1000, 2000, 5000
        flavor="seurat_v3"
    )

# Subset
adata_hvg = test_adata[:, test_adata.var["highly_variable"]].copy()
adata_hvg.X = adata_hvg.X.tocsr()

print(f"Genes: {test_adata.n_vars} → {adata_hvg.n_vars}")
print(f"Expected speedup: ~{test_adata.n_vars / adata_hvg.n_vars:.1f}x")


In [ ]:
adata_hvg.X = adata_hvg.X.copy()
if hasattr(adata_hvg.X, 'data'):
    adata_hvg.X.data = adata_hvg.X.data.copy()

# Load model with HVG subset
torch.set_num_threads(15)   # safe anytime

model.to_device("cpu")

# Prepare sample info
print("Preparing sample info...")
model.sample_info["VOmax26_bin_male"] = (
    model.sample_info["VOmax26_bin_male"].astype("category")
)

# Run DE
print(f"Running DE on {adata_hvg.n_vars} HVGs (this will take ~5-10 min)...")
start = time.time()

de_res = model.differential_expression(
    sample_cov_keys=["VOmax26_bin_male"],
    store_lfc=True,
    mc_samples=50,  # or n_samples=100 if that's correct
    use_vmap=False,
    batch_size=128
)

end = time.time()
runtime = end - start
print(f"\n✓ DE runtime: {runtime/60:.1f} minutes ({runtime:.0f} seconds)")

# Save
output_file = "de_results_VOmax_hvgs.csv"
de_res.to_csv(output_file)
print(f"✓ Results saved to {output_file}")
print(f"✓ Found DE results for {len(de_res)} genes")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import time
import torch

# --------------------------------------------------
# Patient-level VO2max
# --------------------------------------------------

patient_vo2 = (
    adata.obs[["replicate", "VOmax26"]]
    .dropna()
    .drop_duplicates(subset=["replicate"])
    .set_index("replicate")["VOmax26"]
    .astype(float)
)

print("Number of patients:", patient_vo2.shape[0])

# Quantile thresholds
q20, q60, q80, q99 = patient_vo2.quantile([0.20, 0.60, 0.80, 0.99])

print("VO₂max thresholds:")
print(f"q20={q20:.2f}, q60={q60:.2f}, q80={q80:.2f}, q99={q99:.2f}")

# Plot histogram
plt.figure(figsize=(8, 5))
plt.hist(patient_vo2, bins=30)

plt.axvline(q20, linestyle="--", linewidth=2)
plt.axvline(q60, linestyle="--", linewidth=2)
plt.axvline(q80, linestyle="--", linewidth=2)
plt.axvline(q99, linestyle="--", linewidth=2)

plt.xlabel("VO₂max (VOmax26)")
plt.ylabel("Number of patients")
plt.title("VO₂max distribution across patients")

ymax = plt.ylim()[1]
plt.text(q20, ymax * 0.9, "q20", rotation=90, va="top")
plt.text(q60, ymax * 0.9, "q60", rotation=90, va="top")
plt.text(q80, ymax * 0.9, "q80", rotation=90, va="top")
plt.text(q99, ymax * 0.9, "q99", rotation=90, va="top")

plt.tight_layout()
plt.show()


In [ ]:
# Assign group per patient
patient_groups = patient_vo2.apply(vo2_group)

# Map back to cells (DO NOT subset model.adata)
model.adata.obs["VOmax_group"] = (
    model.adata.obs["replicate"]
    .map(patient_groups)
    .astype("category")
)

# Set reference level
model.adata.obs["VOmax_group"] = (
    model.adata.obs["VOmax_group"]
    .cat.reorder_categories(
        ["control_q20_q60", "test_q80_q99"],
        ordered=True
    )
)

# Sanity check
print(model.adata.obs["VOmax_group"].value_counts(dropna=False))


In [ ]:
model.sample_info["VOmax_group"] = (
    model.sample_info["VOmax_group"].astype("category")
)


In [ ]:
import scanpy as sc
from scvi.external.mrvi_torch import TorchMRVI

# Load full data

# Identify highly variable genes (if not already done)
if "highly_variable" not in test_adata.var.columns:
    print("Identifying highly variable genes...")
    sc.pp.highly_variable_genes(test_adata, n_top_genes=2000)

# Subset to HVGs
adata_hvg = test_adata[:, test_adata.var["highly_variable"]].copy()
print(f"Reduced from {test_adata.n_vars} to {adata_hvg.n_vars} genes")


# Prepare
model.sample_info["VOmax26_bin_male"] = model.sample_info["VOmax26_bin_male"].astype("category")

# Run DE (much less memory needed!)
print("Running DE on HVGs only...")
de_res = model.differential_expression(
    sample_cov_keys=["VOmax26_bin_male"],
    store_lfc=True,
    use_vmap=False,
    batch_size=16,
    mc_samples=50
)

de_res.to_csv("de_results_hvgs.csv")
print("✓ Done!")

In [ ]:
import torch

def find_device(obj, visited=None):
    if visited is None:
        visited = set()

    obj_id = id(obj)
    if obj_id in visited:
        return None
    visited.add(obj_id)

    # If the object IS a tensor → we found the device
    if isinstance(obj, torch.Tensor):
        return obj.device

    # If the object has attributes, search them
    if hasattr(obj, "__dict__"):
        for v in obj.__dict__.values():
            try:
                d = find_device(v, visited)
                if d is not None:
                    return d
            except Exception:
                pass

    # If the object is a list/tuple/dict, search inside
    if isinstance(obj, (list, tuple)):
        for v in obj:
            d = find_device(v, visited)
            if d is not None:
                return d

    if isinstance(obj, dict):
        for v in obj.values():
            d = find_device(v, visited)
            if d is not None:
                return d

    return None

print("MRVI model is on:", find_device(model))


In [ ]:
import numpy as np
import pandas as pd

# -----------------------------
# 1. Extract gene names
# -----------------------------
gene = de_res["gene"].values   # shape (5000,)

# -----------------------------
# 2. Extract cell × gene LFC matrix
# -----------------------------
# lfc originally has shape (cells, 1, genes)
lfc_cell_gene = np.squeeze(de_res["lfc"].values)  # (710k, 5000)

# -----------------------------
# 3. Compute gene-level LFC
# -----------------------------
gene_lfc = lfc_cell_gene.mean(axis=0)  # (5000,)

# -----------------------------
# 4. Compute gene-level effect size
#     We use the average absolute LFC per gene
# -----------------------------
effect_size_gene = np.mean(np.abs(lfc_cell_gene), axis=0)  # (5000,)

# -----------------------------
# 5. Build gene-level DE table
# -----------------------------
de_df = pd.DataFrame({
    "gene": gene,
    "lfc": gene_lfc,
    "effect_size": effect_size_gene
})

# -----------------------------
# 6. Get the top 50 DE genes by |LFC|
# -----------------------------
top50 = de_df.reindex(
    de_df["lfc"].abs().sort_values(ascending=False).index
).head(50)

top50


In [ ]:
import numpy as np

# Compute quantiles with your variable names
qc1 = adata.obs["bmi26"].quantile(0.20)
qc2 = adata.obs["bmi26"].quantile(0.60)
qt1 = adata.obs["bmi26"].quantile(0.80)
qt2 = adata.obs["bmi26"].quantile(1.00)

# Create new categorical variable
def assign_bmi_categ(x):
    if qc1 <= x <= qc2:
        return "control"
    elif qt1 <= x <= qt2:
        return "test"
    else:
        return np.nan  # Outside the ranges

adata.obs["bmi_categ"] = adata.obs["bmi26"].apply(assign_bmi_categ)


In [ ]:
adata.obs["bmi_categ"] = adata.obs["bmi_categ"].astype("category")
adata.obs["bmi_categ"] = adata.obs["bmi_categ"].cat.reorder_categories(
    ["control", "test"]
)
adata = adata[~adata.obs["bmi_categ"].isna()].copy()
adata.obs["bmi_categ"].value_counts()



In [ ]:
MRVI.setup_anndata(
    adata,
    sample_info_cols=["bmi_categ"],
)


In [ ]:
de_res = model.differential_expression(
    sample_cov_keys=["bmi_categ"],
    store_lfc=True,
)


In [ ]:
sample_cov_keys = ["Status"]  # Replace with your sample covariate of interest
model.sample_info["Status"] = model.sample_info["Status"].cat.reorder_categories(
    ["Healthy", "Covid"]
)  # Reorder categories such that the coefficient corresponds to Covid
de_res = model.differential_expression(
    sample_cov_keys=sample_cov_keys,
    store_lfc=True,
)